In [5]:
import os, sys, json, glob, time
sys.path.insert(0, os.path.expanduser('~/easyf1/app'))
import engine
BASE = os.path.expanduser('~/easyf1')

In [ ]:

# RAG metrics using the NEUTRAL evaluator (Nemotron Super 120B)
# OpenRouter keeps the evaluator neutral to both contestants.

import os, json, requests, time
from dotenv import load_dotenv

BASE = os.path.expanduser('~/easyf1')
load_dotenv(dotenv_path=os.path.join(BASE, '.env'))
OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY")

# Neutral evaluator — same model we used as the LLM judge
EVAL_MODEL = "nvidia/nemotron-3-super-120b-a12b:free"

def _eval_json(prompt, retries=2):
    """Call the neutral evaluator via OpenRouter and parse JSON robustly."""
    for attempt in range(retries + 1):
        resp = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={"Authorization": f"Bearer {OPENROUTER_KEY}", "Content-Type": "application/json"},
            data=json.dumps({"model": EVAL_MODEL,
                             "messages": [{"role": "user", "content": prompt}]})
        )
        if resp.status_code == 200 and resp.json().get('choices'):
            raw = resp.json()['choices'][0]['message']['content']
            try:
                start = raw.find('{'); end = raw.rfind('}') + 1
                return json.loads(raw[start:end])
            except Exception:
                if attempt < retries:
                    time.sleep(3); continue
                return None
        elif resp.status_code == 429 and attempt < retries:
            time.sleep(15); continue
        else:
            if attempt < retries:
                time.sleep(3); continue
    return None


def faithfulness(answer, context_chunks):
    context = "\n".join(f"- {c}" for c in context_chunks)
    prompt = f"""You are evaluating whether an answer is grounded in the provided context.

CONTEXT:
{context}

ANSWER:
{answer}

Break the answer into its distinct factual claims. For each claim, decide if it is SUPPORTED by the context (true) or NOT supported (false).
Respond ONLY with JSON: {{"total_claims": N, "supported_claims": M}}"""
    r = _eval_json(prompt)
    if not r or r.get('total_claims', 0) == 0:
        return None
    return round(r['supported_claims'] / r['total_claims'], 3)


def answer_relevancy(answer, scenario):
    prompt = f"""You are evaluating whether an answer addresses the question.

QUESTION/SCENARIO:
{scenario}

ANSWER:
{answer}

Rate how directly and completely the answer addresses the question on a scale of 0 to 10 (10 = fully addresses it, 0 = irrelevant).
Respond ONLY with JSON: {{"relevancy_score": N}}"""
    r = _eval_json(prompt)
    if not r or 'relevancy_score' not in r:
        return None
    return round(r['relevancy_score'] / 10, 3)


def context_recall(context_chunks, key_factors):
    context = "\n".join(f"- {c}" for c in context_chunks)
    factors = "\n".join(f"- {f}" for f in key_factors)
    prompt = f"""You are checking whether retrieved context contains the information needed.

RETRIEVED CONTEXT:
{context}

KEY FACTORS THAT SHOULD BE COVERED:
{factors}

For each key factor, decide if the retrieved context contains information relevant to it (true) or not (false).
Respond ONLY with JSON: {{"total_factors": N, "covered_factors": M}}"""
    r = _eval_json(prompt)
    if not r or r.get('total_factors', 0) == 0:
        return None
    return round(r['covered_factors'] / r['total_factors'], 3)


print("Metric functions ready (neutral evaluator) ✅")
print(f"Evaluator: {EVAL_MODEL}")

Metric functions ready (neutral evaluator) ✅
Evaluator: nvidia/nemotron-3-super-120b-a12b:free


In [ ]:

# Testing the metrics on ONE scenario first

import json, glob

BASE = os.path.expanduser('~/easyf1')
results_dir = os.path.join(BASE, 'results')

# Loading eval dataset + ground truth
eval_files = sorted(glob.glob(os.path.join(results_dir, 'eval_dataset_20_*.json')))
with open(eval_files[-1]) as f:
    eval_data = json.load(f)

gt_files = sorted(glob.glob(os.path.join(results_dir, 'ground_truth_20_*.json')))
with open(gt_files[-1]) as f:
    gt_list = json.load(f)
gt_by_num = {g['scenario_num']: g for g in gt_list}

# Taking scenario 1 as the test
item = eval_data[0]
gt = gt_by_num[item['scenario_num']]

print(f"Testing on scenario {item['scenario_num']}")
print(f"Scenario: {item['scenario'][:80]}...\n")

#Faithfulness (per model)
print("Computing faithfulness...")
f_gem = faithfulness(item['gemini'], item['context'])
f_nem = faithfulness(item['nemotron'], item['context'])
print(f"  Gemini:   {f_gem}")
print(f"  Nemotron: {f_nem}")

# --- Answer relevancy (per model) ---
print("\nComputing answer relevancy...")
r_gem = answer_relevancy(item['gemini'], item['scenario'])
r_nem = answer_relevancy(item['nemotron'], item['scenario'])
print(f"  Gemini:   {r_gem}")
print(f"  Nemotron: {r_nem}")

# --- Context recall (retrieval quality — same for both, shared context) ---
print("\nComputing context recall...")
recall = context_recall(item['context'], gt['key_factors'])
print(f"  Context recall: {recall}")

print("\n✅ Single-scenario test complete")

Testing on scenario 1
Scenario: It's lap 32 of 52 at the 2023 British Grand Prix (Silverstone). Lewis Hamilton i...

Computing faithfulness...
  Gemini:   0.429
  Nemotron: 0.167

Computing answer relevancy...
  Gemini:   0.8
  Nemotron: 1.0

Computing context recall...
  Context recall: 1.0

✅ Single-scenario test complete


In [ ]:

# Computing all three metrics across all 20 scenarios
import time
from datetime import datetime

rag_metrics = []

for item in eval_data:
    num = item['scenario_num']
    gt = gt_by_num[num]

    row = {'scenario_num': num}

    # Faithfulness (per model)
    row['faithfulness_gemini']   = faithfulness(item['gemini'], item['context'])
    row['faithfulness_nemotron'] = faithfulness(item['nemotron'], item['context'])

    # Answer relevancy (per model)
    row['relevancy_gemini']   = answer_relevancy(item['gemini'], item['scenario'])
    row['relevancy_nemotron'] = answer_relevancy(item['nemotron'], item['scenario'])

    # Context recall (shared retrieval — one score)
    row['context_recall'] = context_recall(item['context'], gt['key_factors'])

    rag_metrics.append(row)

    # Progress
    print(f"  Scenario {num}: "
          f"faith G={row['faithfulness_gemini']} N={row['faithfulness_nemotron']} | "
          f"relev G={row['relevancy_gemini']} N={row['relevancy_nemotron']} | "
          f"recall={row['context_recall']}")

    time.sleep(3)  # rate-limit courtesy

# Save
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
rag_path = os.path.join(results_dir, f'rag_metrics_{ts}.json')
with open(rag_path, 'w') as f:
    json.dump(rag_metrics, f, indent=2)

print(f"\n{'='*55}")
print(f"RAG metrics computed for {len(rag_metrics)} scenarios ✅")
print(f"Saved: {os.path.basename(rag_path)}")

  Scenario 1: faith G=0.333 N=0.167 | relev G=1.0 N=1.0 | recall=1.0
  Scenario 2: faith G=0.167 N=0.2 | relev G=0.8 N=0.9 | recall=0.0
  Scenario 3: faith G=0.4 N=0.5 | relev G=0.9 N=1.0 | recall=0.75
  Scenario 4: faith G=0.0 N=0.286 | relev G=0.9 N=1.0 | recall=0.0
  Scenario 5: faith G=0.0 N=0.333 | relev G=0.9 N=1.0 | recall=0.0
  Scenario 6: faith G=0.25 N=0.25 | relev G=0.7 N=1.0 | recall=0.5
  Scenario 7: faith G=0.5 N=0.667 | relev G=0.8 N=0.6 | recall=0.25
  Scenario 8: faith G=0.75 N=0.429 | relev G=0.9 N=1.0 | recall=0.25
  Scenario 9: faith G=0.25 N=0.4 | relev G=0.8 N=1.0 | recall=0.25
  Scenario 10: faith G=0.286 N=None | relev G=None N=None | recall=None
  Scenario 11: faith G=None N=None | relev G=None N=None | recall=None
  Scenario 12: faith G=None N=None | relev G=None N=None | recall=None
  Scenario 13: faith G=None N=None | relev G=None N=None | recall=None
  Scenario 14: faith G=None N=None | relev G=None N=None | recall=None
  Scenario 15: faith G=None N=None | 

In [ ]:

# Saving the RAGAS scenarios that are complete
import json
from datetime import datetime

results_dir = os.path.join(BASE, 'results')

# Keep only scenarios where ALL fields are filled (no None)
complete = []
for num in sorted(by_num.keys()):
    r = by_num[num]
    if all(v is not None for k, v in r.items() if k != 'scenario_num'):
        complete.append(r)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
final_path = os.path.join(results_dir, f'rag_metrics_final_{ts}.json')
with open(final_path, 'w') as f:
    json.dump(complete, f, indent=2)

print(f"Complete RAGAS scenarios: {len(complete)}")
print(f"Scenario numbers: {[r['scenario_num'] for r in complete]}")
print(f"Saved: {os.path.basename(final_path)}")

Complete RAGAS scenarios: 9
Scenario numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Saved: rag_metrics_final_20260913_184218.json


In [ ]:

# Analyse the RAGAS metrics of the 9 scenarios 

import json, glob
import pandas as pd

results_dir = os.path.join(BASE, 'results')
rag_files = sorted(glob.glob(os.path.join(results_dir, 'rag_metrics_final_*.json')))
with open(rag_files[-1]) as f:
    rag = json.load(f)

df = pd.DataFrame(rag)

print("="*55)
print(f"RAG METRICS — {len(df)} scenarios")
print("="*55)

print("\nFAITHFULNESS (claims grounded in context):")
print(f"  Gemini:   {df['faithfulness_gemini'].mean():.3f}")
print(f"  Nemotron: {df['faithfulness_nemotron'].mean():.3f}")

print("\nANSWER RELEVANCY (addresses the question):")
print(f"  Gemini:   {df['relevancy_gemini'].mean():.3f}")
print(f"  Nemotron: {df['relevancy_nemotron'].mean():.3f}")

print("\nCONTEXT RECALL (retrieval covered key factors):")
print(f"  Shared:   {df['context_recall'].mean():.3f}")

# Save a tidy summary
summary = {
    'n_scenarios': len(df),
    'faithfulness_gemini': round(df['faithfulness_gemini'].mean(), 3),
    'faithfulness_nemotron': round(df['faithfulness_nemotron'].mean(), 3),
    'relevancy_gemini': round(df['relevancy_gemini'].mean(), 3),
    'relevancy_nemotron': round(df['relevancy_nemotron'].mean(), 3),
    'context_recall': round(df['context_recall'].mean(), 3),
}
with open(os.path.join(results_dir, 'rag_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved summary ✅")

RAG METRICS — 9 scenarios

FAITHFULNESS (claims grounded in context):
  Gemini:   0.294
  Nemotron: 0.359

ANSWER RELEVANCY (addresses the question):
  Gemini:   0.856
  Nemotron: 0.944

CONTEXT RECALL (retrieval covered key factors):
  Shared:   0.333

Saved summary ✅
